In [4]:
"""
Convert 8-bit GeoTIFF RGB files to float32 GeoTIFFs scaled from 0-255 to 0-1.

Input:
    Path to a folder containing .tif / .tiff files.

Output:
    A new sibling folder with the same name plus "_float".
    Example:
        input folder:  D:/data/rgb_tiles
        output folder: D:/data/rgb_tiles_float
"""

from pathlib import Path

import numpy as np
import rasterio


def convert_tif_to_float32(input_tif: Path, output_tif: Path, overwrite: bool = False) -> None:
    """
    Convert one GeoTIFF from uint8 0-255 to float32 0-1.
    """

    if output_tif.exists() and not overwrite:
        print(f"Skipping existing file: {output_tif}")
        return

    with rasterio.open(input_tif) as src:
        profile = src.profile.copy()

        if src.dtypes[0] != "uint8":
            print(f"Warning: {input_tif.name} is {src.dtypes[0]}, not uint8. It will still be divided by 255.")

        if src.count < 3:
            print(f"Warning: {input_tif.name} has {src.count} band(s), expected RGB with 3 bands.")

        # Read all bands and scale to 0-1
        data = src.read().astype(np.float32) / 255.0

        # Update output profile
        profile.update(
            dtype="float32",
            compress="lzw",
            predictor=3,
            BIGTIFF="IF_SAFER"
        )

        output_tif.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(output_tif, "w", **profile) as dst:
            dst.write(data)

    print(f"Created: {output_tif}")


def convert_folder(input_folder: Union[Path, str], output_folder: Union[Path, str] = None) -> Path:
    """
    Convert all tif/tiff files in a folder.
    """

    input_folder = Path(input_folder)

    if not input_folder.exists():
        raise FileNotFoundError(f"Input folder does not exist: {input_folder}")

    if not input_folder.is_dir():
        raise NotADirectoryError(f"Input path is not a folder: {input_folder}")
    
    output_folder = Path(output_folder) if output_folder else None

    if not output_folder:
        output_folder = input_folder.parent / f"{input_folder.name}_float"
    
    output_folder.mkdir(parents=True, exist_ok=True)

    tif_files = sorted(
        list(input_folder.glob("*.tif")) +
        list(input_folder.glob("*.tiff"))
    )

    if not tif_files:
        print(f"No .tif or .tiff files found in: {input_folder}")
        return output_folder

    print(f"Input folder:  {input_folder}")
    print(f"Output folder: {output_folder}")
    print(f"Files found:   {len(tif_files)}")

    for tif_path in tif_files:
        output_path = output_folder / tif_path.name
        convert_tif_to_float32(tif_path, output_path)

    return output_folder

In [8]:
pth_out = convert_folder(r"r:\BiH_ALS_2025\BiH_ALS_2025_DMO_05m_e4MSTP", r"r:\BiH_ALS_2025\e4MSTP_float")
pth_out

Input folder:  r:\BiH_ALS_2025\BiH_ALS_2025_DMO_05m_e4MSTP
Output folder: r:\BiH_ALS_2025\e4MSTP_float
Files found:   1026
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4756166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4758166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4760166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4762166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4764166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4766166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4768166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4770166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4772166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4774166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4776166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\241726_4778166_rvt_e4MSTP.tif
Created: r:\BiH_ALS_2025\e4MSTP_float\243726_4750166_rvt_e4MS

WindowsPath('r:/BiH_ALS_2025/e4MSTP_float')

In [1]:
from osgeo import gdal
from pathlib import Path

gdal.UseExceptions()


def build_vrt_overviews(
    vrt_path,
    levels=(4, 8, 16, 32, 64, 128, 256, 512, 1024),
    resampling="average",
    threads="ALL_CPUS",
    compress="LZW",
    blocksize=512,
):
    """
    Build external pyramids / overviews for a VRT.

    Output:
        <vrt_path>.ovr

    For continuous rasters such as DEM, DFM, SLRM:
        resampling="average" or "bilinear"

    For categorical rasters / masks:
        resampling="nearest" or "mode"
    """

    vrt_path = Path(vrt_path)

    if not vrt_path.exists():
        raise FileNotFoundError(vrt_path)

    # Important for large VRT overviews
    gdal.SetConfigOption("GDAL_NUM_THREADS", str(threads))
    gdal.SetConfigOption("COMPRESS_OVERVIEW", compress)
    gdal.SetConfigOption("BIGTIFF_OVERVIEW", "IF_SAFER")
    gdal.SetConfigOption("GDAL_TIFF_OVR_BLOCKSIZE", str(blocksize))

    # Useful for LZW/DEFLATE/ZSTD overviews.
    # Use 2 for integer data, 3 for floating-point data.
    gdal.SetConfigOption("PREDICTOR_OVERVIEW", "2")

    # Optional, useful if large nodata areas exist
    gdal.SetConfigOption("SPARSE_OK_OVERVIEW", "ON")

    # Open read-only to force external overviews.
    # For VRT this is normally the correct mode.
    ds = gdal.Open(vrt_path.as_posix(), gdal.GA_ReadOnly)
    if ds is None:
        raise RuntimeError(f"Could not open VRT: {vrt_path}")

    print(f"Building overviews for: {vrt_path}")
    print(f"Overview levels: {levels}")

    ds.BuildOverviews(
        resampling.upper(),
        list(levels),
        callback=gdal.TermProgress_nocb,
    )

    ds = None  # close dataset and flush .ovr

    ovr_path = vrt_path.with_suffix(vrt_path.suffix + ".ovr")
    print(f"Done: {ovr_path}")

    return ovr_path

In [2]:
build_vrt_overviews(
    r"r:\BiH_ALS_2025\e4MSTP_float.vrt",
    levels=(4, 8, 16, 32, 64, 128, 256, 512, 1024),
    resampling="bilinear",
)

Building overviews for: r:\BiH_ALS_2025\e4MSTP_float.vrt
Overview levels: (4, 8, 16, 32, 64, 128, 256, 512, 1024)
Done: r:\BiH_ALS_2025\e4MSTP_float.vrt.ovr


WindowsPath('r:/BiH_ALS_2025/e4MSTP_float.vrt.ovr')

In [3]:
mis1 = r"r:\delovno\nejc\26-6_missing_e4\mis1.gpkg"
mis2 = r"r:\delovno\nejc\26-6_missing_e4\mis2.gpkg"

import geopandas as gpd

gdf1 = gpd.read_file(mis1)
gdf2 = gpd.read_file(mis2)

In [6]:
gdf1

,MINX,MINY,MAXX,MAXY,CNTX,CNTY,AREA,PERIM,HEIGHT,WIDTH,geometry
0,253498.464926,4.758999e+06,254001.11965,4.759501e+06,253749.792288,4.759250e+06,252203.323378,2008.794788,501.742671,502.654723,"MULTIPOLYGON (((252498.465 4757998.896, 252498..."


In [7]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import from_bounds


def save_raster_window_from_bounds(
    raster_path,
    output_path,
    minx,
    miny,
    maxx,
    maxy,
    *,
    indexes=None,
    boundless=True,
    fill_value=None,
    compress="lzw",
    predictor=None,
):
    """
    Read a raster window from explicit map-coordinate bounds and save it as a GeoTIFF.

    Parameters
    ----------
    raster_path : str or pathlib.Path
        Input raster path. Can be GeoTIFF, VRT, or any rasterio-readable raster.
    output_path : str or pathlib.Path
        Output GeoTIFF path.
    minx, miny, maxx, maxy : float
        Window bounds in the same CRS as the raster.
    indexes : int, list[int], or None
        Band index or indices to read. If None, all bands are read.
    boundless : bool
        If True, allows reading outside raster bounds and fills missing pixels.
    fill_value : number or None
        Fill value for boundless reads. If None, uses raster nodata if defined,
        otherwise 0.
    compress : str or None
        GeoTIFF compression.
    predictor : int or None
        TIFF predictor. Use 2 for integer data, 3 for float data.

    Returns
    -------
    pathlib.Path
        Path to saved GeoTIFF.
    """

    raster_path = Path(raster_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(raster_path) as src:
        window = from_bounds(
            minx,
            miny,
            maxx,
            maxy,
            transform=src.transform,
        )

        if fill_value is None:
            fill_value = src.nodata if src.nodata is not None else 0

        out_image = src.read(
            indexes=indexes,
            window=window,
            boundless=boundless,
            fill_value=fill_value,
        )

        # If a single band is read as 2D, convert to rasterio write shape: (bands, rows, cols)
        if out_image.ndim == 2:
            out_image = out_image[np.newaxis, :, :]

        out_transform = src.window_transform(window)
        out_profile = src.profile.copy()

        out_profile.update(
            driver="GTiff",
            height=out_image.shape[1],
            width=out_image.shape[2],
            count=out_image.shape[0],
            transform=out_transform,
            compress=compress,
        )

        if predictor is not None:
            out_profile.update(predictor=predictor)

        with rasterio.open(output_path, "w", **out_profile) as dst:
            dst.write(out_image)

    return output_path

In [10]:
save_raster_window_from_bounds(
    raster_path=r"r:\BiH_ALS_2025\BiH_ALS_2025_DMO_05m.vrt",
    output_path=r"r:\delovno\nejc\26-6_missing_e4\mis2.tif",
    minx=265726.5,
    miny=4808166,
    maxx=271726.5,
    maxy=4814166,
    fill_value=0,
    predictor=3,
)

WindowsPath('r:/delovno/nejc/26-6_missing_e4/mis2.tif')